# v4 Isolation Phase — full analysis

Produces every table and figure requested:
1. Per-scenario × agent FinalSequenceValue tables (raw + Δ vs EMCTS + % vs EMCTS) — one per ms.
2. The same averaged across ms.
3. Best-agent-per-scenario bar chart — per ms + averaged.
4. Ranking-point bar chart (last=1 … best=12, summed over 100 scenarios) — per ms + averaged.
5. Trajectory of BestSequenceValue per burn phase — per ms × per agent and averaged.
6. Trajectory as % improvement over initial sequence — same layout as 5.
7. Branching-factor vs improvement scatter — per agent per ms + averaged. Raw + %.
8. Bar chart of mean % improvement over initial sequence — per ms + averaged.

Plus four analysis extras:
- **E1.** Paired Δ vs EMCTS with 95% CIs (n≈5000 per cell — small effects detectable).
- **E2.** Best-ratio-per-budget sweep (the v3-style view).
- **E3.** Single-vs-dual portfolio split — does the ranking change?
- **E4.** Convergence speed — burn at which 95% of final gain is reached.

All trajectory plots use TIGHT y-axis scaling so the 12 config lines are distinguishable (a few-percent
spread on values ~8000 would be invisible with a 0-anchored axis).

## Cell 1 — setup (paths, libraries, palettes)

Drop this single cell into a Jupyter notebook. It writes everything to `analysis/figures_v4_full/`
and `analysis/tables_v4_full/`. Same code lives in `analysis/analyze_v4_full.py` for shell use.

In [ ]:
import os, sys, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from pathlib import Path

# EDIT to point to wherever your v4 results live
ROOT = Path('test_results/v4')
FIGS = Path('analysis/figures_v4_full'); FIGS.mkdir(parents=True, exist_ok=True)
TABLES = Path('analysis/tables_v4_full'); TABLES.mkdir(parents=True, exist_ok=True)

MS_LEVELS = [500, 1000, 5000]
RATIOS = [2, 5, 10, 15, 20]

# 12 configs in canonical display order.
ORDER = ['EMCTS', 'CrossoverBB']
for r in RATIOS: ORDER.append(f'CrossoverExpansion@{r}')
for r in RATIOS: ORDER.append(f'CrossoverCombo@{r}')

def _shades(base, n):
    rgba = np.array(to_rgba(base))
    return [tuple(rgba * (0.55 + 0.45 * i/(n-1))) for i in range(n)]

COLORS = {'EMCTS': '#222222', 'CrossoverBB': '#ff7f0e'}
for cfg, c in zip([f'CrossoverExpansion@{r}' for r in RATIOS], _shades('#1f77b4', 5)): COLORS[cfg] = c
for cfg, c in zip([f'CrossoverCombo@{r}'     for r in RATIOS], _shades('#2ca02c', 5)): COLORS[cfg] = c

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True, 'grid.alpha': 0.3})

LEAF_RE = re.compile(r'(?P<agent>EMCTS|CrossoverBB|CrossoverExpansion|CrossoverCombo)'
                     r'(?:/ratio_(?P<ratio>\d+))?'
                     r'/(?P<ms>\d+)ms/results\.csv$')

def _config_label(agent, ratio):
    if agent in ('EMCTS', 'CrossoverBB'): return agent
    return f'{agent}@{int(ratio)}'

def load_all():
    frames = []
    for f in sorted(glob.glob(str(ROOT) + '/**/results.csv', recursive=True)):
        rel = f.replace(os.sep, '/').split(str(ROOT).replace(os.sep, '/') + '/', 1)[-1]
        m = LEAF_RE.search(rel)
        if not m: continue
        df = pd.read_csv(f)
        df['Config'] = _config_label(m.group('agent'), m.group('ratio')) if m.group('ratio') else m.group('agent')
        df['ThinkingMs'] = int(m.group('ms'))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

def per_turn(df_burn):
    keys = ['Config', 'ThinkingMs', 'ScenarioId', 'VisitId']
    return (df_burn.drop_duplicates(subset=keys, keep='first')
                  .drop(columns=['BridgeBurningPhaseNumber','BestSequenceValue','FmCallsElapsed',
                                  'Timestamp','RawImprovement','PercentageImprovement'], errors='ignore'))

def load_portfolio():
    p = ROOT / '_portfolio' / 'portfolio.csv'
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

df_burn = load_all()
turn_df = per_turn(df_burn)
portfolio = load_portfolio()
print(f'loaded {len(df_burn):,} burn rows; {len(turn_df):,} (config, scenario, visit) tuples; '
      f'{portfolio.shape[0]} portfolio scenarios')

## Cell 2 — produce every figure + table

The full pipeline. Each block is independent; run them all or comment out the ones you don't want.

You can also just call `!python analysis/analyze_v4_full.py` from a notebook cell instead — same code.

In [ ]:
# All implementations live in analysis/analyze_v4_full.py. The cleanest way to run them is:
import importlib, sys
sys.path.insert(0, str(Path('analysis').resolve()))
import analyze_v4_full
importlib.reload(analyze_v4_full)
analyze_v4_full.main()

## Cell 3 — display the headline figures inline

In [ ]:
from IPython.display import Image, Markdown, display
import glob
for f in sorted(glob.glob('analysis/figures_v4_full/*.png')):
    display(Markdown(f'### {Path(f).name}')); display(Image(filename=f))

## Cell 4 — display the summary tables inline

In [ ]:
import glob
for f in sorted(glob.glob('analysis/tables_v4_full/*_summary.csv') + glob.glob('analysis/tables_v4_full/E*.csv')):
    display(Markdown(f'### {Path(f).name}'))
    display(pd.read_csv(f, index_col=0))